In [ ]:
# Author: Michael Pratt
import os

import imageio
import gymnasium as gym
import numpy as np
import torch
from agilerl.algorithms.dqn_rainbow import RainbowDQN
from agilerl.components.replay_buffer import (
    MultiStepReplayBuffer,
    PrioritizedReplayBuffer,
)
from agilerl.training.train_off_policy import train_off_policy
from agilerl.utils.utils import make_vect_envs
from tqdm import trange

In [ ]:
ENV_NAME = "ALE/Alien-v5"       # ← change this to any Gym discrete env
NUM_ENVS = 8                   # vectorised envs for speed
MAX_STEPS = 150_000

In [ ]:
        INIT_HP = {
            "BATCH_SIZE": 64,  # Batch size
            "LR": 0.0001,  # Learning rate
            "GAMMA": 0.99,  # Discount factor
            "MEMORY_SIZE": 100_000,  # Max memory buffer size
            "LEARN_STEP": 1,  # Learning frequency
            "N_STEP": 3,  # Step number to calculate td error
            "PER": True,  # Use prioritized experience replay buffer
            "ALPHA": 0.6,  # Prioritized replay buffer parameter
            "BETA": 0.4,  # Importance sampling coefficient
            "TAU": 0.001,  # For soft update of target parameters
            "PRIOR_EPS": 0.000001,  # Minimum priority for sampling
            "NUM_ATOMS": 51,  # Unit number of support
            "V_MIN": -200.0,  # Minimum value of support
            "V_MAX": 200.0,  # Maximum value of support
            "NOISY": True,  # Add noise directly to the weights of the network
            "LEARNING_DELAY": 1000,  # Steps before starting learning
            "TARGET_SCORE": 200.0,  # Target score that will beat the environment
            "MAX_STEPS": 200000,  # Maximum number of steps an agent takes in an environment
            "EVO_STEPS": 10000,  # Evolution frequency
            "EVAL_STEPS": None,  # Number of evaluation steps per episode
            "EVAL_LOOP": 1,  # Number of evaluation episodes
        }

In [ ]:

num_envs = 16
env = make_vect_envs("ALE/Alien-v5", num_envs=num_envs)  # Create environment

observation_space = env.single_observation_space
action_space = env.single_action_space

In [ ]:
# Set-up the device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Define the network configuration of a simple mlp with two hidden layers, each with 64 nodes
net_config = {
    "encoder_config": {"hidden_size": [64, 64]},  # Encoder hidden size
    "head_config": {"hidden_size": [64, 64]}  # Head hidden size
}

# Define a Rainbow-DQN agent
rainbow_dqn = RainbowDQN(
    observation_space=observation_space,
    action_space=action_space,
    net_config=net_config,
    batch_size=INIT_HP["BATCH_SIZE"],
    lr=INIT_HP["LR"],
    learn_step=INIT_HP["LEARN_STEP"],
    gamma=INIT_HP["GAMMA"],
    tau=INIT_HP["TAU"],
    beta=INIT_HP["BETA"],
    n_step=INIT_HP["N_STEP"],
    device=device,
)

In [ ]:
memory = PrioritizedReplayBuffer(
    max_size=INIT_HP["MEMORY_SIZE"],
    alpha=INIT_HP["ALPHA"],
    device=device,
)
n_step_memory = MultiStepReplayBuffer(
    max_size=INIT_HP["MEMORY_SIZE"],
    n_step=INIT_HP["N_STEP"],
    gamma=INIT_HP["GAMMA"],
    device=device,
)

In [ ]:
# Define parameters per and n_step


def train_agent():

    trained_pop, pop_fitnesses = train_off_policy(
        env=env,
        env_name="CartPole-v1",
        algo="RainbowDQN",
        pop=[rainbow_dqn],
        memory=memory,
        n_step_memory=n_step_memory,
        INIT_HP=INIT_HP,
        max_steps=INIT_HP["MAX_STEPS"],
        evo_steps=INIT_HP["EVO_STEPS"],
        eval_steps=INIT_HP["EVAL_STEPS"],
        eval_loop=INIT_HP["EVAL_LOOP"],
        learning_delay=INIT_HP["LEARNING_DELAY"],
        target=INIT_HP["TARGET_SCORE"],
        n_step=True,
        per=True,
        tournament=None,
        mutation=None,
        wb=False,  # Boolean flag to record run with Weights & Biases
        checkpoint=INIT_HP["MAX_STEPS"],
        checkpoint_path="RainbowDQN.pt",
    )

In [ ]:

if __name__ == "__main__":
    print("Training started...")
    train_agent()

In [ ]:
# rainbow_dqn = RainbowDQN.load(save_path, device=device)